In [1]:
#loading metacells of rna and atac data
import anndata as anndata
import pandas as pd
import numpy as np

sc_rna = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_annot_RNA.h5ad")  
sc_atac = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_annot_ATAC.h5ad")

window_sizes = ["100kb"]
base_path = "/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/20 windows"

combined_gene_peaks = {}
for window in window_sizes:
    combined_gene_peaks[window] = pd.read_csv(f"{base_path}/combined_gene_peak_assignments_{window}.csv")

In [2]:
sc_rna.obs_names.unique().size, sc_atac.obs_names.unique().size

(1260, 1260)

In [3]:
#function for normalizing the aggregated data by total counts per SEACell to get relative accessibility/expression values (compositional normalization)
def compositional_normalize_adata(adata):
    import pandas as pd
    import scipy.sparse as sp
    
    # extract to dense DataFrame
    if sp.issparse(adata.X):
        df = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    else:
        df = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
    
    # normalize
    row_sums = df.sum(axis=1)
    if (row_sums == 0).any():
        print(f"Warning: {(row_sums == 0).sum()} metacells with zero counts.")
    df_norm = df.div(row_sums, axis=0)
    
    # put back into AnnData, preserving obs/var metadata
    adata_norm = adata.copy()
    adata_norm.X = df_norm.values
    return adata_norm

sc_rna_norm = compositional_normalize_adata(sc_rna)
sc_atac_norm = compositional_normalize_adata(sc_atac)

In [4]:
#Reindex the dataframes to ensure they are in the same order
sc_atac_norm = sc_atac_norm[sc_rna_norm.obs_names, :]
sc_rna_norm = sc_rna_norm[sc_atac_norm.obs_names, :]
print(sc_atac_norm.obs_names.equals(sc_rna_norm.obs_names))  # Should return True

sc_rna_norm.shape

True


(1260, 32057)

In [5]:
#Substract the genes that are present in the gene_peaks_10kb dataframe from the sc_rna_norm dataframe, to only keep the genes that have peaks assigned to them
combined_gene_peaks_100kb = combined_gene_peaks["100kb"]["gene_id"].tolist()
sc_rna_norm = sc_rna_norm[:, combined_gene_peaks_100kb] 
print(sc_rna_norm.shape)

(1260, 19380)


In [6]:
#taking the minimum non-zero value in the sc_rna_norm matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np
non_zero_mask = (sc_rna_norm.X > 0)
epsilon_rna = np.min(sc_rna_norm.X[non_zero_mask])

#taking the minimum non-zero value in the sc_atac_norm matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (sc_atac_norm.X > 0)
epsilon_at= np.min(sc_atac_norm.X[non_zero_mask])

In [7]:
#log transformation (for norm. distribution) & scaling 10000000+1
#log scaling both datasets by multiplying by 10 million and adding 1 to avoid log(0) issues, then taking log10
sc_rna_norm.X = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000) 
sc_atac_norm.X = np.log10((sc_atac_norm.X+ epsilon_at)*10000000)
print(sc_atac_norm.X)
print(sc_rna_norm.X)

/home/fgsasse_lrs_1/miniforge3/envs/seacells/lib/python3.12/site-packages/anndata/_core/anndata.py:639: FutureWarning: Setting element `.X` of view of `AnnData` object will obey copy-on-write semantics in the next minor release. 
  if self._handle_view_X_cow(value):
/tmp/ipykernel_3448507/3560636608.py:3: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  sc_rna_norm.X = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000)
/tmp/ipykernel_3448507/3560636608.py:4: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  sc_atac_norm.X = np.log10((sc_atac_norm.X+ epsilon_at)*10000000)


[[1.22406574 1.42506063 1.42506063 ... 0.22324167 2.36609084 2.35511303]
 [0.22324167 1.48706397 1.48706397 ... 1.48706397 2.42668356 2.4197507 ]
 [1.09377419 1.60164548 1.30756139 ... 0.22324167 2.38135032 2.36822775]
 ...
 [0.22324167 0.22324167 0.22324167 ... 0.22324167 2.56909164 2.57686894]
 [0.22324167 0.22324167 0.22324167 ... 0.22324167 2.48359112 2.47769107]
 [0.22324167 0.22324167 0.22324167 ... 2.33067662 2.78118271 2.77227791]]
[[2.31285294 1.6915537  2.85523383 ... 3.22914718 2.98836083 2.96008914]
 [2.22024575 2.22024575 2.90994656 ... 3.02467639 2.90994656 3.04495294]
 [2.18499897 2.37166614 2.88340025 ... 2.88340025 2.83120931 3.02295613]
 ...
 [1.6915537  1.6915537  1.6915537  ... 1.6915537  3.4837788  1.6915537 ]
 [1.6915537  1.6915537  3.07111186 ... 1.6915537  3.18088045 1.6915537 ]
 [1.6915537  1.6915537  1.6915537  ... 1.6915537  1.6915537  3.5049896 ]]


In [8]:
combined_gene_peaks["100kb"].head()

,gene_id,assigned_peaks
0,a1cf,"['12-6186644-6187111', '12-6190483-6191143', '..."
1,a2ml,"['15-21143884-21144162', '15-21033797-21034001..."
2,aaas,"['9-323370-323691', '9-209853-211311', '9-2304..."
3,aacs,"['5-18949828-18950713', '5-18811696-18812645',..."
4,aadac,"['15-1052335-1053001', '15-1029184-1030074', '..."


## Normal OLS on 100kb window (bidirectional)

In [9]:
#check the maximum number of peaks assigned to each gene in all windows
import ast  
print("Max peaks assigned to a gene in 100kb window:", combined_gene_peaks["100kb"]["assigned_peaks"].apply(lambda x: len(ast.literal_eval(x)) if pd.notnull(x) else 0).max())

#number of samples in the pseudobulk datasets
print(sc_rna_norm.n_obs)
print(sc_atac_norm.n_obs)

Max peaks assigned to a gene in 100kb window: 125
1260
1260


In [11]:
#computing OLS regression for 100kb window by loading the function from the src folder 
from pathlib import Path
import sys

src_path = Path("/home/fgsasse_lrs_1/Downloads/BA/src/")

if str(src_path) not in sys.path:
       sys.path.insert(0, str(src_path)),

#import several functions from the src folder to classify peak-gene pairs and aggregate results
from your_package.ols import (
    compute_peak_gene_ols,
    classify_peak_gene_pairs,
    aggregate_peak_gene_categories)

ols_results_100kb = {}
ols_dfs_100kb = {}
agg_ols_dfs_100kb = {}


gene_peak_dict_100kb = {window: combined_gene_peaks[window] for window in window_sizes}

window_label = ["100kb"]

for window in window_label:

    print(f"Computing OLS regression for {window} window...")

    # print max memory used
    import tracemalloc
    tracemalloc.start()

    ols_results_100kb[window], ols_dfs_100kb[window] = compute_peak_gene_ols(
        rna_data=sc_rna_norm,
        atac_data=sc_atac_norm,
        gene_peaks_df=gene_peak_dict_100kb[window],
        window_label=window,
    )

    # report max memory
    current, peak = tracemalloc.get_traced_memory()
    print(f"compute_peak_gene_ols memory usage: {current / 10**6:.2f} MB; Peak memory usage: {peak / 10**6:.2f} MB")
    tracemalloc.stop()

    ols_dfs_100kb[window] = classify_peak_gene_pairs(
        ols_dfs_100kb[window]
    )

    agg_ols_dfs_100kb[window] = aggregate_peak_gene_categories(
        ols_dfs_100kb[window]
    )

agg_ols_dfs_100kb[window].head()

#save the OLS results for the 100kb window to a csv file
ols_dfs_100kb["100kb"].to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/OLS/Results/ols_results_100kb_df_sc.csv", index=False)


Computing OLS regression for 100kb window...
[100kb] OLS complete — 19380 genes processed
OLS DataFrame: 1,871,887 peak–gene pairs
compute_peak_gene_ols memory usage: 941.92 MB; Peak memory usage: 1414.13 MB


## Plotting the OLS results

In [ ]:
#read in the saved all_ols_dfs dataframe to confirm it was saved correctly
ols_100kb_df = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/OLS_results/ols_results_100kb_df_sc.csv")
ols_100kb_df.head()

#aggregate the number of significant peaks per gene and category  
agg_ols_100kb_df = (
    ols_100kb_df
    .dropna(subset=["pval", "coef", "category"])
    .groupby(["window", "gene", "category"]) 
    .size()
    .reset_index(name="count")
)

In [ ]:
agg_ols_100kb_df.head()

## Deduplication across windows

In [ ]:
#read in the saved all_ols_dfs dataframe to confirm it was saved correctly
import pandas as pd
ols_all_df = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/OLS/Results/ols_res_all_windows_df_sc.csv")
print(ols_all_df.head())

#make window categorical and ordered
ols_all_df["window"] = pd.Categorical(
    ols_all_df["window"],
    categories=["10kb", "20kb", "50kb", "100kb"],
    ordered=True
)

# Order by window
ols_all_df = ols_all_df.sort_values("window",ascending=True)
print(ols_all_df[["window", "gene", "peak", "coef", "pval", "category"]].head())

#"Dedublication" of gene-peak pairs by keeping only the first occurrence of each pair across all windows (since they are ordered by window size, this will keep the pair with the smallest window size)"
pair_summary = (
    ols_all_df
    .groupby(["gene", "peak"], as_index=False)
    .first()
)

#Sort by window to keep the smallest window size for each gene-peak pair
pair_summary = pair_summary.sort_values("window", ascending=True)
print(pair_summary[["window", "gene", "peak", "coef", "pval", "category"]].head())

#Sanity check to confirm that each gene-peak pair is now unique
pair_counts = pair_summary.groupby(["gene", "peak"]).size().reset_index(name="count")
print(pair_counts['count'].value_counts()) # Should show that all pairs have a count of 1

#Counts how many unique gene-peak pairs fall into each category for each window size
summary_counts = (pair_summary.groupby(["window", "gene", "category"]).size().reset_index(name="count"))
summary_counts.head()

In [ ]:
#Boxplot of the counts of peaks per gene (first occurrence of gene-peak pair across all windows) for each category and window size
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

category_order  = ["sig. negative", "non-significant", "sig. positive"]
category_colors = {
    "sig. negative":   "#D43737",
    "sig. positive":   "#1A8EB9",
    "non-significant": "#A9A9A9",
}

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("#F8F8F8")

sns.boxplot(
    data=summary_counts,
    x="window", y="count", hue="category",
    hue_order=category_order,
    palette=category_colors,
    width=0.6,
    linewidth=1.2,
    fill=True,
    flierprops=dict(marker="o", markersize=4, linewidth=0),
    boxprops=dict(alpha=1),
    gap=0.1,
    ax=ax,
)

# Replace x-axis tick labels
window_labels = {
    "10kb":  "±TSS–10 kb",
    "20kb":  "±10–20 kb",
    "50kb":  "±20–50 kb",
    "100kb": "±50–100 kb",
}
ax.set_xticklabels([window_labels[t.get_text()] for t in ax.get_xticklabels()])

# Recolor fliers manually to match their box color
for line, color in zip(
    [c for c in ax.get_lines() if c.get_linestyle() == "none"],
    [category_colors[cat]
     for _ in summary_counts["window"].cat.categories
     for cat in category_order]
):
    line.set_markerfacecolor(color)
    line.set_markeredgecolor(color)
    line.set_alpha(0.7)

ax.set_title("Peak counts per gene across genomic windows (SEACells)",
             fontsize=14, fontweight="500", pad=14, loc="center")
ax.set_xlabel("Genomic window", fontsize=11, labelpad=8)
ax.set_ylabel("# significant peaks per gene", fontsize=11, labelpad=8)

ax.yaxis.grid(True, color="#cccccc", linewidth=0.8, linestyle="-", zorder=0)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#a5a4a4")

handles = [mpatches.Patch(color=category_colors[c], alpha=1, label=c.capitalize())
           for c in category_order]
ax.legend(
    handles=handles,
    title="Coefficient Category\n(pval ≤ 0.05)",
    frameon=True,
    fontsize=10,
    loc="upper left",
    title_fontsize=10,
)

plt.tight_layout()
plt.show()